In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
import xgboost as xgb
from xgboost import XGBClassifier
import optuna
# from tabpfn import TabPFNClassifier

In [5]:
def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

In [10]:
train_df = pd.read_csv("../train.csv")
test_df = pd.read_csv("../test.csv")
train_df.drop(train_df.columns[0], axis = 1)
test_df.drop(test_df.columns[0], axis = 1)

,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_Score_mean,T1_FGM,T1_FGA,...,T2_opponent_Blk,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO,CoachELO
0,2024.0,NaN,1101.0,NaN,1102.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024.0,NaN,1101.0,NaN,1103.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024.0,NaN,1101.0,NaN,1104.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024.0,NaN,1101.0,NaN,1105.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024.0,NaN,1101.0,NaN,1106.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129956,2024.0,NaN,3475.0,NaN,3477.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129957,2024.0,NaN,3475.0,NaN,3478.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129958,2024.0,NaN,3476.0,NaN,3477.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129959,2024.0,NaN,3476.0,NaN,3478.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
train = train_df.copy()
test = test_df.copy()

Modifications

In [7]:
train.columns

Index(['Unnamed: 0', 'Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID',
       'T2_Score', 'location', 'T1_Score_mean', 'T1_FGM', 'T1_FGA', 'T1_FGM3',
       'T1_FGA3', 'T1_FTM', 'T1_FTA', 'T1_OR', 'T1_DR', 'T1_Ast', 'T1_TO',
       'T1_Stl', 'T1_Blk', 'T1_PF', 'T1_opponent_Score', 'T1_opponent_FGM',
       'T1_opponent_FGA', 'T1_opponent_FGM3', 'T1_opponent_FGA3',
       'T1_opponent_FTM', 'T1_opponent_FTA', 'T1_opponent_OR',
       'T1_opponent_DR', 'T1_opponent_Ast', 'T1_opponent_TO',
       'T1_opponent_Stl', 'T1_opponent_Blk', 'T1_opponent_PF', 'T1_PointDiff',
       'T2_Score_mean', 'T2_FGM', 'T2_FGA', 'T2_FGM3', 'T2_FGA3', 'T2_FTM',
       'T2_FTA', 'T2_OR', 'T2_DR', 'T2_Ast', 'T2_TO', 'T2_Stl', 'T2_Blk',
       'T2_PF', 'T2_opponent_Score', 'T2_opponent_FGM', 'T2_opponent_FGA',
       'T2_opponent_FGM3', 'T2_opponent_FGA3', 'T2_opponent_FTM',
       'T2_opponent_FTA', 'T2_opponent_OR', 'T2_opponent_DR',
       'T2_opponent_Ast', 'T2_opponent_TO', 'T2_opponent_Stl',
     

In [ ]:
x_train, y_train = x_y_from_data_frame()

,Unnamed: 0,Season,DayNum,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_Score_mean,T1_FGM,...,T2_opponent_Blk,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,TeamELO,CoachELO
0,0,2003,134,1421,92,1411,84,0,83.000000,28.000000,...,2.20,22.800000,7.200000,1.000000,1.000000,16,16,0,1324.742776,1445.589445
1,1,2003,136,1112,80,1436,51,0,83.000000,30.333333,...,3.00,17.000000,12.000000,0.666667,1.000000,1,16,-15,2087.437190,2125.806920
2,2,2003,136,1113,84,1272,71,0,82.333333,30.666667,...,2.00,20.000000,7.250000,0.666667,0.750000,10,7,3,1810.546112,1843.235904
3,3,2003,136,1141,79,1166,73,0,83.400000,25.400000,...,3.00,17.666667,8.666667,1.000000,1.000000,11,6,5,1704.724089,1729.865982
4,4,2003,136,1143,76,1301,74,0,63.666667,24.000000,...,2.80,21.000000,0.000000,0.333333,0.600000,8,9,-1,1903.105267,1931.203185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2649,2649,2023,146,1400,81,1274,88,0,70.200000,24.800000,...,4.00,14.666667,-1.000000,0.800000,0.666667,2,5,-3,2088.892590,2004.918895
2650,2650,2023,146,1166,56,1361,57,0,82.500000,28.250000,...,3.75,17.000000,10.000000,0.750000,1.000000,6,5,1,1999.557485,2012.656581
2651,2651,2023,152,1274,59,1163,72,0,76.666667,28.333333,...,1.50,14.500000,11.500000,0.666667,0.750000,5,4,1,2050.257792,2091.700370
2652,2652,2023,152,1194,71,1361,72,0,80.000000,29.000000,...,3.75,17.000000,10.000000,1.000000,1.000000,9,5,4,1840.208418,1939.283841
